In [ ]:
import os
import gc
import sys
import json
import re
import warnings
from pathlib import Path

import torch
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import logging as hf_logging

N = 100          
SAVE_CSV = True  

warnings.filterwarnings("ignore")
hf_logging.set_verbosity_error()
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

ROOT = Path("/root/autodl-tmp")
CODE_ROOT = ROOT / "code" / "OpenOneRec"
DATA_ROOT = ROOT / "data" / "OpenOneRec" / "hf_raw" / "OpenOneRec-RecIF" / "benchmark_data"
VIDEO_FILE = DATA_ROOT / "video" / "video_test.parquet"
MODEL_ROOT = ROOT / "checkpoints" / "OpenOneRec" / "OneRec-1.7B"
TEMPLATE_FILE = CODE_ROOT / "benchmarks" / "benchmark" / "tasks" / "v1_0" / "qwen3_soft_switch.jinja2"
RUN_DIR = ROOT / "runs" / "OpenOneRec" / "exp_video_recall32_repro"
RUN_DIR.mkdir(parents=True, exist_ok=True)

# 基本检查
assert "onerec-lite" in sys.executable, f"当前不是 onerec-lite 内核: {sys.executable}"
assert torch.cuda.is_available(), "CUDA 不可用"
assert "sm_120" in torch.cuda.get_arch_list(), f"当前 torch 不支持 sm_120: {torch.cuda.get_arch_list()}"
assert VIDEO_FILE.exists(), VIDEO_FILE
assert MODEL_ROOT.exists(), MODEL_ROOT
assert TEMPLATE_FILE.exists(), TEMPLATE_FILE


for _name in ["model", "tokenizer"]:
    if _name in globals():
        del globals()[_name]
gc.collect()
torch.cuda.empty_cache()

# Read Data
df = pd.read_parquet(VIDEO_FILE)
if N is None:
    N = len(df)
else:
    N = min(int(N), len(df))


SID_BLOCK_PATTERN = re.compile(r"<\|sid_begin\|>.*?<\|sid_end\|>")
CORE_SID_PATTERN = re.compile(r"<s_a_\d+><s_b_\d+><s_c_\d+>")

def flatten_content_to_text(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict) and item.get("type") == "text":
                parts.append(item.get("text", ""))
            else:
                parts.append(str(item))
        return "".join(parts)
    return str(content)

def normalize_messages(messages):
    out = []
    for msg in messages:
        out.append({
            "role": msg.get("role", "user"),
            "content": flatten_content_to_text(msg.get("content", "")),
        })
    return out

def extract_first_core_sid(text):
    m = CORE_SID_PATTERN.search(text or "")
    return m.group(0) if m else None

def normalize_gt_sid(text):
    m = CORE_SID_PATTERN.search(text or "")
    return m.group(0) if m else None

# tokenizer / model
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ROOT,
    trust_remote_code=True,
)
tokenizer.chat_template = TEMPLATE_FILE.read_text(encoding="utf-8")
if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ROOT,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
)
model = model.to("cuda")
model.eval()

for _attr in ["temperature", "top_p", "top_k"]:
    if hasattr(model.generation_config, _attr):
        try:
            setattr(model.generation_config, _attr, None)
        except Exception:
            pass

rows = []
for i in tqdm(range(N), desc=f"Evaluating first {N} video samples"):
    row = df.iloc[i]
    metadata = json.loads(row["metadata"]) if isinstance(row["metadata"], str) else row["metadata"]
    messages = json.loads(row["messages"]) if isinstance(row["messages"], str) else row["messages"]

    norm_messages = normalize_messages(messages)

    prompt = tokenizer.apply_chat_template(
        norm_messages,
        tokenize=False,
        add_generation_prompt=True,
    ) + "<|sid_begin|>"

    # ground truth
    gt_blocks = SID_BLOCK_PATTERN.findall(metadata["answer"])
    gt_ids = [normalize_gt_sid(x) for x in gt_blocks]
    gt_ids = [x for x in gt_ids if x is not None]

    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=True)
    inputs = {k: v.to("cuda") for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=3,
            do_sample=False,
            num_beams=32,
            num_return_sequences=32,
            early_stopping=True,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
        )

    prompt_len = inputs["input_ids"].shape[1]
    decoded = tokenizer.batch_decode(outputs[:, prompt_len:], skip_special_tokens=False)

    pred_ids = [extract_first_core_sid(text) for text in decoded]
    pred_ids = [x for x in pred_ids if x is not None]
    top32 = pred_ids[:32]

    hits = sum(1 for gt in gt_ids if gt in top32)
    recall32 = hits / len(gt_ids) if gt_ids else 0.0
    pass32 = any(gt in top32 for gt in gt_ids) if gt_ids else False
    position1_pass32 = (gt_ids[0] in top32) if gt_ids else False

    rows.append({
        "row_idx": i,
        "num_gt": len(gt_ids),
        "num_pred": len(pred_ids),
        "hits": hits,
        "recall32": recall32,
        "pass32": pass32,
        "position1_pass32": position1_pass32,
        "top5_pred": pred_ids[:5],
        "top5_gt": gt_ids[:5],
        "uid": metadata.get("uid"),
        "uuid": metadata.get("uuid"),
    })

    del inputs, outputs, decoded
    if (i + 1) % 20 == 0:
        torch.cuda.empty_cache()

res = pd.DataFrame(rows)

print("\n===== SUMMARY =====")
print("N =", N)
print("mean recall@32 =", res["recall32"].mean())
print("mean pass@32 =", res["pass32"].mean())
print("mean position1_pass@32 =", res["position1_pass32"].mean())

if SAVE_CSV:
    out_csv = RUN_DIR / f"video_recall32_first_{N}.csv"
    res.to_csv(out_csv, index=False)
    print("saved to:", out_csv)

res.head(20)


libgomp: Invalid value for environment variable OMP_NUM_THREADS
/root/miniconda3/envs/onerec-lite/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Evaluating first 100 video samples: 100%|██████████| 100/100 [02:23<00:00,  1.44s/it]


===== SUMMARY =====
N = 100
mean recall@32 = 0.023123015873015875
mean pass@32 = 0.13
mean position1_pass@32 = 0.06
saved to: /root/autodl-tmp/runs/OpenOneRec/exp_video_recall32_repro/video_recall32_first_100.csv


,row_idx,num_gt,num_pred,hits,recall32,pass32,position1_pass32,top5_pred,top5_gt,uid,uuid
0,0,10,32,0,0.000000,False,False,"[<s_a_718><s_b_7592><s_c_2091>, <s_a_5158><s_b...","[<s_a_2398><s_b_1901><s_c_5357>, <s_a_3313><s_...",11901,ab60c761-c20a-4aaa-95ed-f37ab7a56dce
1,1,10,32,1,0.100000,True,True,"[<s_a_4922><s_b_2117><s_c_4310>, <s_a_3561><s_...","[<s_a_4922><s_b_2117><s_c_4310>, <s_a_7832><s_...",3502,51d6547b-2569-4e05-bcef-660b4001c2f0
2,2,10,32,0,0.000000,False,False,"[<s_a_6942><s_b_1600><s_c_2798>, <s_a_2387><s_...","[<s_a_2947><s_b_1114><s_c_5581>, <s_a_5494><s_...",13503,1d18c713-54b8-4e84-8663-2e1920014f1f
3,3,10,32,0,0.000000,False,False,"[<s_a_851><s_b_7126><s_c_5511>, <s_a_7470><s_b...","[<s_a_3271><s_b_1302><s_c_4350>, <s_a_4902><s_...",7260,e795d9ba-fa48-4d93-9943-1b97ae801da6
4,4,10,32,1,0.100000,True,True,"[<s_a_4660><s_b_238><s_c_889>, <s_a_3675><s_b_...","[<s_a_3675><s_b_6998><s_c_4552>, <s_a_4660><s_...",7057,c09503d0-8651-41ec-8122-43c381eae53e
5,5,9,32,0,0.000000,False,False,"[<s_a_539><s_b_2728><s_c_3997>, <s_a_4603><s_b...","[<s_a_5009><s_b_1473><s_c_8008>, <s_a_5006><s_...",3719,f3ea5ec6-a5a6-423f-912d-49460cc7d61a
6,6,10,32,0,0.000000,False,False,"[<s_a_771><s_b_5115><s_c_8103>, <s_a_5627><s_b...","[<s_a_2463><s_b_5929><s_c_6933>, <s_a_1897><s_...",8334,39404633-022b-4faf-ac71-d1b299858492
7,7,8,32,0,0.000000,False,False,"[<s_a_1811><s_b_7884><s_c_4579>, <s_a_2702><s_...","[<s_a_3524><s_b_6353><s_c_6319>, <s_a_6392><s_...",8294,82118b2c-3d67-459d-b5ec-e9b72827ec63
8,8,10,32,0,0.000000,False,False,"[<s_a_2974><s_b_3740><s_c_3204>, <s_a_429><s_b...","[<s_a_6414><s_b_5813><s_c_5544>, <s_a_6414><s_...",15763,20fdf47b-6300-40d2-91b5-2f67fd433a53
9,9,6,32,0,0.000000,False,False,"[<s_a_6717><s_b_487><s_c_5600>, <s_a_2964><s_b...","[<s_a_7309><s_b_6511><s_c_7433>, <s_a_6117><s_...",11996,eaf2969d-1489-409b-b108-556ac55ae01a
